# 📝 Azure AI Search Index 생성

이 노트북에서는 Azure AI Search의 인덱스(Index)를 생성합니다.

## 📋 학습 목표

1. **Index 구조 이해하기** - 필드(Field) 타입과 속성 학습
2. **Vector Search 설정하기** - HNSW 알고리즘을 활용한 벡터 검색 구성
3. **Semantic Search 설정하기** - 의미론적 검색을 위한 구성
4. **Index 생성 및 확인하기** - Azure AI Search 서비스에 인덱스 배포

## 🔍 주요 개념

### Index란?
- 검색 가능한 문서들의 집합
- 관계형 데이터베이스의 '테이블'과 유사한 개념
- 각 문서는 JSON 형태로 저장

### Field 타입
- **SimpleField**: 단순 필드 (필터링, 정렬 가능)
- **SearchableField**: 전체 텍스트 검색 가능한 필드
- **SearchField**: 벡터 검색을 위한 필드

### Vector Search
- **HNSW 알고리즘**: 고속 근사 최근접 이웃 검색
- **Cosine Similarity**: 벡터 간 유사도 측정 방식

## 📦 SDK 버전 정보

현재 사용 중인 SDK: **azure-search-documents >= 11.6.0**  
최신 GA 버전: **azure-search-documents 12.0.0** (2026-05-01, service API 2026-04-01)

✅ 최신 GA 버전 사용 중

---

## 1️⃣ 라이브러리 임포트

Azure AI Search Index 생성에 필요한 라이브러리를 임포트합니다.

**⏱️ 참고:** Azure SDK 라이브러리는 크기가 커서 첫 임포트 시 10-30초 정도 소요될 수 있습니다. 이는 정상입니다!

In [1]:
print("⏳ 라이브러리 임포트 중... (10-30초 소요)")

from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch
)
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
import os

print("✅ 라이브러리 임포트 완료!")
print(f"   - azure-search-documents 버전 확인 완료")
print(f"   - 필요한 모든 모듈 로드 완료")

⏳ 라이브러리 임포트 중... (10-30초 소요)
✅ 라이브러리 임포트 완료!
   - azure-search-documents 버전 확인 완료
   - 필요한 모든 모듈 로드 완료


## 2️⃣ 환경 변수 로드 및 클라이언트 초기화

`.env` 파일에서 Azure AI Search 접속 정보를 로드하고 SearchIndexClient를 생성합니다.

In [2]:
# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 접속 정보
endpoint = os.getenv("SEARCH_ENDPOINT")
key = os.getenv("SEARCH_ADMIN_KEY")
index_name = os.getenv("SEARCH_INDEX_NAME")

if not endpoint or not key:
    print("❌ 환경 변수가 설정되지 않았습니다. .env 파일을 확인하세요.")
else:
    print("✅ 환경 변수 로드 완료")
    print(f"   Endpoint: {endpoint}")
    print(f"   Index Name: {index_name}")
    
    # SearchIndexClient 생성
    credential = AzureKeyCredential(key)
    client = SearchIndexClient(endpoint=endpoint, credential=credential)
    print("✅ SearchIndexClient 생성 완료")

✅ 환경 변수 로드 완료
   Endpoint: https://ai-search-foundry-iq-cj.search.windows.net
   Index Name: products-index
✅ SearchIndexClient 생성 완료


## 3️⃣ Index Fields 정의

검색 인덱스의 필드(Field)를 정의합니다. 각 필드는 특정 속성(searchable, filterable, facetable 등)을 가질 수 있습니다.

### 필드 속성 설명:
- **key**: 문서의 고유 식별자 (각 인덱스에 하나만 존재)
- **searchable**: 전체 텍스트 검색 가능
- **filterable**: OData 필터 쿼리 사용 가능
- **sortable**: 정렬 가능
- **facetable**: 패싯 집계 가능 (카테고리별 카운트 등)
- **analyzer_name**: 텍스트 분석기 (한국어: "ko.lucene")

### 💡 데이터셋 컬럼:
- **id**: 상품 고유 ID
- **title**: 상품명
- **brand**: 브랜드
- **category**: 카테고리
- **normal_price**: 정상가
- **image_link**: 이미지 URL
- **review**: 사용후기 (LLM 생성)

In [3]:
fields = [
    # 고유 식별자 (필수)
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True
    ),
    
    # 상품명 (검색 가능, 한국어 분석기)
    SearchableField(
        name="title",
        type=SearchFieldDataType.String,
        analyzer_name="ko.lucene"
    ),
    
    # 브랜드 (검색, 필터, 패싯 가능)
    SearchableField(
        name="brand",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True,
        analyzer_name="ko.lucene"
    ),
    
    # 카테고리 (필터, 패싯 가능)
    SimpleField(
        name="category",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True
    ),
    
    # 가격 (필터, 정렬 가능)
    SimpleField(
        name="normal_price",
        type=SearchFieldDataType.Int32,
        filterable=True,
        sortable=True
    ),
    
    # 사용후기 (검색 가능, 한국어 분석기)
    SearchableField(
        name="review",
        type=SearchFieldDataType.String,
        analyzer_name="ko.lucene"
    )
]

print(f"✅ {len(fields)}개의 필드 정의 완료")
for field in fields:
    print(f"   - {field.name} ({field.type})")


✅ 6개의 필드 정의 완료
   - id (Edm.String)
   - title (Edm.String)
   - brand (Edm.String)
   - category (Edm.String)
   - normal_price (Edm.Int32)
   - review (Edm.String)


## 6️⃣ 기존 Index 삭제 (선택사항)

노트북을 여러 번 실행하거나 Index를 처음부터 다시 만들 때 사용합니다.

**⚠️ 주의:** 이 셀을 실행하면 기존 인덱스와 모든 데이터가 삭제됩니다!

In [4]:
# 기존 인덱스가 있는지 확인하고 삭제
try:
    # 인덱스 목록 조회
    existing_indexes = [idx.name for idx in client.list_indexes()]
    
    if index_name in existing_indexes:
        print(f"⚠️  기존 Index '{index_name}'가 발견되었습니다.")
        
        # 사용자 확인 (실제 운영에서는 입력받는 것이 안전함)
        # 노트북에서는 자동으로 삭제하도록 설정
        delete_existing = True  # False로 변경하면 삭제하지 않음
        
        if delete_existing:
            client.delete_index(index_name)
            print(f"🗑️  기존 Index '{index_name}' 삭제 완료")
        else:
            print(f"ℹ️  기존 Index를 유지합니다. 다음 단계에서 업데이트됩니다.")
    else:
        print(f"ℹ️  기존 Index '{index_name}'가 없습니다. 새로 생성합니다.")
        
except Exception as e:
    print(f"⚠️  Index 확인 중 오류: {str(e)}")

ℹ️  기존 Index 'products-index'가 없습니다. 새로 생성합니다.


## 7️⃣ Index 생성 및 배포

정의한 필드와 구성을 기반으로 Azure AI Search 서비스에 인덱스를 생성합니다.

In [5]:
print(f"\n{'='*60}")
print(f"📝 Index '{index_name}' 생성 중...")
print(f"{'='*60}\n")

# SearchIndex 객체 생성
index = SearchIndex(
    name=index_name,
    fields=fields
)

try:
    # Index 생성 또는 업데이트
    result = client.create_or_update_index(index)
    
    print(f"✅ Index '{index_name}' 생성 성공!\n")
    print(f"📊 Index 정보:")
    print(f"   필드 수: {len(result.fields)}")
    print(f"\n🔗 Azure Portal에서 확인:")
    print(f"   {endpoint}")
    
except Exception as e:
    print(f"❌ Index 생성 실패: {str(e)}")


📝 Index 'products-index' 생성 중...

✅ Index 'products-index' 생성 성공!

📊 Index 정보:
   필드 수: 6

🔗 Azure Portal에서 확인:
   https://ai-search-foundry-iq-cj.search.windows.net


## 8️⃣ Index 상세 정보 확인

생성된 인덱스의 세부 정보를 확인합니다.

In [6]:
try:
    # Index 조회
    retrieved_index = client.get_index(index_name)
    
    print(f"\n📋 Index 상세 정보: '{index_name}'\n")
    print("=" * 60)
    
    # 필드 목록
    print("\n🔹 필드 목록:")
    for field in retrieved_index.fields:
        field_info = []
        if hasattr(field, 'key') and field.key:
            field_info.append("key")
        if hasattr(field, 'searchable') and field.searchable:
            field_info.append("searchable")
        if hasattr(field, 'filterable') and field.filterable:
            field_info.append("filterable")
        if hasattr(field, 'sortable') and field.sortable:
            field_info.append("sortable")
        if hasattr(field, 'facetable') and field.facetable:
            field_info.append("facetable")
        
        attributes = f" [{', '.join(field_info)}]" if field_info else ""
        print(f"   • {field.name} ({field.type}){attributes}")
    
    print("\n" + "=" * 60)
    print("✅ Index 상세 정보 조회 완료")
    
except Exception as e:
    print(f"❌ Index 조회 실패: {str(e)}")


📋 Index 상세 정보: 'products-index'


🔹 필드 목록:
   • id (Edm.String) [key, filterable]
   • title (Edm.String) [searchable]
   • brand (Edm.String) [searchable, filterable, facetable]
   • category (Edm.String) [filterable, facetable]
   • normal_price (Edm.Int32) [filterable, sortable]
   • review (Edm.String) [searchable]

✅ Index 상세 정보 조회 완료


---

## 🎯 다음 단계

Index 생성이 완료되었습니다! 이제 다음 단계로 진행하세요:

1. **upload_data.ipynb** - 상품 데이터를 인덱스에 업로드
2. **03-search** - 다양한 검색 방법 실습

---

## 💡 참고 사항

### Index 수정 시 주의사항
- 일부 필드 속성은 기존 인덱스에서 변경 불가 (예: key, 데이터 타입)
- 구조적 변경이 필요한 경우 인덱스를 삭제하고 재생성해야 함

### Vector Search 최적화 팁
- **m 값**: 4-10 사이 권장 (기본값 4)
- **efConstruction**: 100-1000 사이 권장 (기본값 400)
- **efSearch**: 100-1000 사이 권장 (기본값 500)
- 데이터셋 크기와 정확도 요구사항에 따라 조정

### Semantic Search 요구사항
- Azure AI Search의 **Basic** 이상 SKU 필요 (Free tier에서는 사용 불가)
- 추가 비용 발생 (쿼리당 과금)

---

## 🔗 관련 문서

- [Azure AI Search Index 개념](https://learn.microsoft.com/azure/search/search-what-is-an-index)
- [Vector Search 가이드](https://learn.microsoft.com/azure/search/vector-search-overview)
- [Semantic Search 개요](https://learn.microsoft.com/azure/search/semantic-search-overview)
- [HNSW 알고리즘 상세](https://learn.microsoft.com/azure/search/vector-search-ranking)

---

## 🧭 다음 단계

| ⬅️ 이전 | 🏠 목차 | ➡️ 다음 |
|:---------|:-------:|--------:|
| [Lab 01: 연결 테스트](../01-setup/test_connection.ipynb) | [워크숍 홈](../README.md) | [Lab 02-2: 데이터 업로드](02-upload_data.ipynb) |